# Notebook 05: Final Evaluation with Judgy

## Learning Goals
- Understand why judge bias matters
- Learn how statistical bias correction works
- Use the `judgy` library to get corrected success rates
- Calculate and interpret confidence intervals
- Write your final analysis

## The Problem: Judge Bias

Imagine your judge has:
- **TPR = 0.9** (90% of PASS examples are correctly identified)
- **TNR = 0.8** (80% of FAIL examples are correctly identified)

Now you run the judge on 1000 new traces and it says: **"850 PASS, 150 FAIL"**

### Question: What's the TRUE success rate?

**It's NOT 85%!** Why?

- The judge **misses 10% of PASS** examples (calls them FAIL)
- The judge **misses 20% of FAIL** examples (calls them PASS)

These errors **bias** the observed success rate. The `judgy` library mathematically corrects for this!

### The Math (Simplified):

```
Observed rate = (TPR × true_success_rate) + ((1 - TNR) × (1 - true_success_rate))
```

We can solve for `true_success_rate` if we know TPR and TNR (which we measured in Notebook 04!).

### Why This Matters:
- **Raw judge output is biased** - don't trust it directly
- **Statistical correction** gives you the true underlying rate
- **Confidence intervals** tell you how certain you can be

This is the **key innovation** that makes LLM-as-Judge reliable!

## Step 1: Setup and Install Judgy

In [ ]:
# Install judgy if needed
!pip install judgy -q

print("✅ judgy installed")

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from judgy import estimate_success_rate

print("✅ Libraries imported")

## Step 2: Load Your Test Set Metrics

We need the TPR and TNR from your test set evaluation (Notebook 04).

In [ ]:
# Load test metrics from Notebook 04
with open('../results/test_metrics.json', 'r') as f:
    test_metrics = json.load(f)

TPR = test_metrics['TPR']
TNR = test_metrics['TNR']

print("📊 Test Set Performance:")
print(f"="*50)
print(f"TPR (True Positive Rate):  {TPR:.3f} ({TPR*100:.1f}%)")
print(f"TNR (True Negative Rate):  {TNR:.3f} ({TNR*100:.1f}%)")
print(f"Accuracy:                  {test_metrics['accuracy']:.3f}")
print(f"="*50)

print(f"\n📝 What this means:")
print(f"   • The judge correctly identifies {TPR*100:.1f}% of actually-PASS recipes")
print(f"   • The judge correctly identifies {TNR*100:.1f}% of actually-FAIL recipes")
print(f"   • These values will be used to correct bias in our final evaluation")

## Step 3: Run Judge on Full Dataset

Now we'll run your judge on the full `raw_traces.csv` dataset (~2400 traces).

### Important Notes:
- This will make ~2400 API calls - could cost $20-50 depending on your model choice
- Could take 20-40 minutes with rate limiting
- You can use a smaller sample (e.g., 500 traces) to save costs while learning

### Option: Use Reference Implementation Results
If you want to skip the API calls, you can use the reference implementation's results
from `../../results/final_evaluation.json` for learning purposes.

In [ ]:
# Load the full raw traces dataset
raw_traces = pd.read_csv('../../data/raw_traces.csv')

print(f"Loaded {len(raw_traces)} raw traces")
print(f"\nFirst few rows:")
raw_traces.head()

In [ ]:
# Choose how many to evaluate
# Options:
# - Full dataset: len(raw_traces)  (~2400, expensive)
# - Large sample: 1000
# - Medium sample: 500
# - Small sample: 100 (for quick testing)

SAMPLE_SIZE = 100  # Change this based on your budget
USE_FULL_DATASET = False  # Set to True to use all traces

if USE_FULL_DATASET:
    eval_traces = raw_traces
    print(f"Using full dataset: {len(eval_traces)} traces")
else:
    # Random sample
    eval_traces = raw_traces.sample(n=min(SAMPLE_SIZE, len(raw_traces)), random_state=42)
    print(f"Using sample: {len(eval_traces)} traces")
    print(f"This represents {len(eval_traces)/len(raw_traces)*100:.1f}% of the full dataset")

In [ ]:
RUN_JUDGE_ON_FULL_DATA = False  # Set to True when ready

if not RUN_JUDGE_ON_FULL_DATA:
    print("⚠️ Skipping judge evaluation (RUN_JUDGE_ON_FULL_DATA = False)")
    print("\nSet RUN_JUDGE_ON_FULL_DATA = True when ready.")
    print("\nAlternatively, you can use the reference implementation results")
    print("or create synthetic data for learning purposes.")
else:
    # Import the judge function from Notebook 04
    from litellm import completion
    import time
    
    # Load judge prompt
    with open('../prompts/judge_prompt_v1.txt', 'r') as f:
        judge_prompt_template = f.read()
    
    MODEL = "gpt-4"  # Use same model as Notebook 04
    
    def call_judge(query, dietary_restriction, response):
        """Call the LLM judge (same function from Notebook 04)."""
        filled_prompt = judge_prompt_template.format(
            query=query,
            dietary_restriction=dietary_restriction,
            response=response
        )
        
        try:
            llm_response = completion(
                model=MODEL,
                messages=[{"role": "user", "content": filled_prompt}],
                temperature=0.0,
            )
            
            response_text = llm_response.choices[0].message.content
            
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            elif "```" in response_text:
                response_text = response_text.split("```")[1].split("```")[0]
            
            result = json.loads(response_text.strip())
            result["label"] = result["label"].upper()
            
            return result
        except Exception as e:
            print(f"Error: {str(e)}")
            return None
    
    print(f"Running judge on {len(eval_traces)} traces...")
    print(f"Estimated time: {len(eval_traces) * 0.5 / 60:.1f} minutes")
    print(f"Estimated cost: ${len(eval_traces) * 0.02:.2f} (rough estimate)\n")
    
    full_predictions = []
    
    for idx, row in eval_traces.iterrows():
        if idx % 50 == 0:
            print(f"Progress: {idx}/{len(eval_traces)}")
        
        result = call_judge(
            query=row['query'],
            dietary_restriction=row['dietary_restriction'],
            response=row['response']
        )
        
        full_predictions.append({
            'query': row['query'],
            'dietary_restriction': row['dietary_restriction'],
            'predicted': result['label'] if result else 'ERROR',
            'reasoning': result['reasoning'] if result else 'API call failed'
        })
        
        time.sleep(0.5)  # Rate limiting
    
    # Save predictions
    full_predictions_df = pd.DataFrame(full_predictions)
    full_predictions_df.to_csv('../results/full_dataset_predictions.csv', index=False)
    
    print(f"\n✅ Complete! Results saved to ../results/full_dataset_predictions.csv")

## Step 4: Calculate Observed Success Rate

First, let's see what the **raw, uncorrected** success rate is.

In [ ]:
# Load predictions (either from your run or create sample data)
try:
    full_predictions = pd.read_csv('../results/full_dataset_predictions.csv')
    print(f"✅ Loaded predictions for {len(full_predictions)} traces")
except FileNotFoundError:
    print("❌ No predictions found.")
    print("Either run Step 3 or use reference implementation results.")
    print("\nFor learning purposes, creating synthetic data...")
    
    # Create synthetic data matching reference implementation pattern
    np.random.seed(42)
    n = 2400
    
    # Simulate predictions with ~85% observed PASS rate
    full_predictions = pd.DataFrame({
        'query': [f'Query {i}' for i in range(n)],
        'dietary_restriction': np.random.choice(['vegan', 'vegetarian', 'gluten-free', 'keto'], n),
        'predicted': np.random.choice(['PASS', 'FAIL'], n, p=[0.857, 0.143]),
        'reasoning': ['Synthetic reasoning' for i in range(n)]
    })
    
    print(f"Created synthetic data for {len(full_predictions)} traces")

In [ ]:
# Calculate observed success rate
full_predictions_clean = full_predictions[full_predictions['predicted'] != 'ERROR'].copy()

n_total = len(full_predictions_clean)
n_pass = (full_predictions_clean['predicted'] == 'PASS').sum()
n_fail = (full_predictions_clean['predicted'] == 'FAIL').sum()

observed_success_rate = n_pass / n_total

print("="*60)
print("OBSERVED (UNCORRECTED) RESULTS")
print("="*60)
print(f"\nTotal traces evaluated: {n_total}")
print(f"Judge said PASS: {n_pass} ({n_pass/n_total*100:.1f}%)")
print(f"Judge said FAIL: {n_fail} ({n_fail/n_total*100:.1f}%)")
print(f"\n📊 Observed Success Rate: {observed_success_rate:.3f} ({observed_success_rate*100:.1f}%)")
print("="*60)

print(f"\n⚠️ But this is BIASED! We need to correct for judge errors.")
print(f"   Remember: TPR = {TPR:.3f}, TNR = {TNR:.3f}")
print(f"   The judge isn't perfect, so we can't trust this raw rate.")

## Step 5: Use Judgy for Bias Correction

Now we'll use the `judgy` library to correct for the judge's bias.

### How Judgy Works:

1. **Input**: 
   - Observed success rate (p_obs)
   - Judge's TPR and TNR
   - Sample size

2. **Mathematical Correction**:
   - Solves for the true underlying success rate (θ)
   - Accounts for false positives and false negatives

3. **Output**:
   - Corrected success rate (θ̂)
   - 95% Confidence interval
   - How much correction was applied

In [ ]:
# Create a BinaryRater with your judge's performance
rater = BinaryRater(
    tpr=TPR,  # True Positive Rate from test set
    tnr=TNR   # True Negative Rate from test set
)

print(f"✅ Created BinaryRater with:")
print(f"   TPR = {TPR:.3f}")
print(f"   TNR = {TNR:.3f}")

# Some stats about the rater
print(f"\n📊 Judge Characteristics:")
print(f"   Accuracy if data is 50/50: {(TPR + TNR) / 2:.3f}")
print(f"   Miss rate for PASS: {1 - TPR:.3f} ({(1-TPR)*100:.1f}%)")
print(f"   Miss rate for FAIL: {1 - TNR:.3f} ({(1-TNR)*100:.1f}%)")

In [ ]:
# Apply bias correction
result = rater.estimate(
    p_obs=observed_success_rate,  # Observed success rate
    n=n_total,                     # Sample size
    confidence=0.95                # 95% confidence interval
)

corrected_rate = result['theta']  # θ̂ (theta-hat) - corrected success rate
ci_lower = result['ci'][0]        # Lower bound of 95% CI
ci_upper = result['ci'][1]        # Upper bound of 95% CI
correction = corrected_rate - observed_success_rate

print("="*60)
print("CORRECTED RESULTS (Using Judgy)")
print("="*60)

print(f"\n📊 Observed Success Rate (Raw):")
print(f"   p_obs = {observed_success_rate:.3f} ({observed_success_rate*100:.1f}%)")

print(f"\n✨ Corrected Success Rate:")
print(f"   θ̂ = {corrected_rate:.3f} ({corrected_rate*100:.1f}%)")

print(f"\n📈 95% Confidence Interval:")
print(f"   [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"   [{ci_lower*100:.1f}%, {ci_upper*100:.1f}%]")

print(f"\n🔧 Correction Applied:")
print(f"   {correction:+.3f} ({correction*100:+.1f} percentage points)")
if correction > 0:
    print(f"   → Judge was too strict (had false negatives)")
elif correction < 0:
    print(f"   → Judge was too lenient (had false positives)")
else:
    print(f"   → No correction needed (perfectly calibrated judge)")

print("="*60)

## Step 6: Visualize the Results

In [ ]:
# Create visualization comparing observed vs corrected
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Observed vs Corrected rates
rates = [observed_success_rate, corrected_rate]
labels = ['Observed\n(Biased)', 'Corrected\n(True Rate)']
colors = ['lightcoral', 'lightgreen']

bars = ax1.bar(labels, rates, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Success Rate')
ax1.set_title('Observed vs Corrected Success Rate')
ax1.set_ylim([0, 1])
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% baseline')

# Add value labels on bars
for bar, rate in zip(bars, rates):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{rate:.1%}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Plot 2: Confidence interval
ax2.barh(['Corrected Rate'], [corrected_rate], color='lightgreen', alpha=0.7, edgecolor='black')
ax2.errorbar([corrected_rate], ['Corrected Rate'], 
             xerr=[[corrected_rate - ci_lower], [ci_upper - corrected_rate]],
             fmt='none', color='black', capsize=10, capthick=2, linewidth=2)
ax2.set_xlabel('Success Rate')
ax2.set_title('95% Confidence Interval')
ax2.set_xlim([0, 1])
ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.text(corrected_rate, 0, f'{corrected_rate:.1%}', 
         ha='center', va='bottom', fontsize=12, fontweight='bold')
ax2.text(ci_lower, -0.15, f'{ci_lower:.1%}', ha='center', fontsize=10)
ax2.text(ci_upper, -0.15, f'{ci_upper:.1%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/final_evaluation_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved to ../results/final_evaluation_visualization.png")

## Step 7: Save Final Results

In [ ]:
# Save comprehensive final results
final_results = {
    'evaluation': {
        'total_traces': int(n_total),
        'sample_or_full': 'sample' if not USE_FULL_DATASET else 'full',
        'n_pass': int(n_pass),
        'n_fail': int(n_fail)
    },
    'judge_performance': {
        'TPR': float(TPR),
        'TNR': float(TNR),
        'accuracy_on_test': float(test_metrics['accuracy'])
    },
    'results': {
        'observed_success_rate': float(observed_success_rate),
        'corrected_success_rate': float(corrected_rate),
        'correction_applied': float(correction),
        'confidence_interval': {
            'lower': float(ci_lower),
            'upper': float(ci_upper),
            'confidence_level': 0.95
        }
    },
    'interpretation': {
        'corrected_rate_pct': f"{corrected_rate*100:.1f}%",
        'ci_pct': f"[{ci_lower*100:.1f}%, {ci_upper*100:.1f}%]",
        'judge_bias': 'too_strict' if correction > 0 else 'too_lenient' if correction < 0 else 'well_calibrated'
    }
}

# Save to JSON
with open('../results/final_evaluation.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("✅ Final results saved to ../results/final_evaluation.json")

# Pretty print
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(json.dumps(final_results, indent=2))
print("="*60)

## Step 8: Write Your Analysis

The homework requires a 1-2 paragraph analysis interpreting your results.

### Questions to Address:
1. **What is the Recipe Bot's true dietary adherence rate?**
   - Use the corrected success rate
   
2. **How confident are you in this estimate?**
   - Discuss the confidence interval
   - Narrow CI = high confidence, wide CI = less confidence
   
3. **What does this mean for users?**
   - Is the bot reliable for people with dietary restrictions?
   - Where should improvements focus?
   
4. **How did bias correction help?**
   - Was the judge too strict or lenient?
   - How much did the correction matter?

### Template:

Write your analysis in the markdown cell below:

### Your Analysis (1-2 paragraphs):

**Paragraph 1: Main Findings**

[Discuss the corrected success rate, confidence interval, and what this means for dietary adherence]

**Paragraph 2: Bias Correction and Implications**

[Discuss the judge's bias (too strict/lenient?), how much correction was applied, and what this means for trusting LLM-as-Judge evaluations]

---

**Example structure:**

"Based on our LLM-as-Judge evaluation of [N] Recipe Bot traces, we found that the bot has a corrected dietary adherence success rate of [X]% (95% CI: [lower]%-[upper]%). This was corrected from an observed rate of [Y]%, accounting for our judge's imperfect TPR of [TPR] and TNR of [TNR]. The [narrow/wide] confidence interval suggests [high/moderate] confidence in this estimate, indicating that...

The statistical bias correction revealed that our judge was [too strict/too lenient], [over/under]estimating the success rate by [Z] percentage points. This highlights the importance of measuring and correcting for judge bias when using LLM-as-Judge methodologies. For users with dietary restrictions, this [X]% success rate means... [discuss implications for product quality and where improvements should focus]."

## Summary: What You've Learned

### Key Concepts:

1. **Judge Bias**: LLM judges aren't perfect - they have measurable error rates
2. **Statistical Correction**: We can mathematically correct for bias using TPR/TNR
3. **Confidence Intervals**: Quantify uncertainty in our estimates
4. **The Judgy Library**: Automates the complex math for bias correction

### The Complete LLM-as-Judge Pipeline:

1. **Generate/collect traces** (HW3 Option 1, or use provided data)
2. **Label ground truth** (manual labeling by domain experts)
3. **Split data** (train/dev/test to prevent overfitting)
4. **Develop judge** (prompt engineering with few-shot examples)
5. **Measure performance** (TPR/TNR on test set)
6. **Run on production data** (evaluate thousands of traces)
7. **Apply bias correction** (get true underlying rate)

### Why This Matters:

You now understand how to:
- **Scale evaluation** from manual (dozens) to automated (thousands)
- **Trust automated evaluation** by measuring and correcting bias
- **Quantify uncertainty** with confidence intervals
- **Make data-driven decisions** about AI system quality

This is a **production-ready evaluation methodology** used by real AI companies!

### Homework Deliverables Checklist:

- ✅ Labeled dataset with train/dev/test splits
- ✅ Final judge prompt with few-shot examples
- ✅ Judge performance metrics (TPR/TNR on test set)
- ✅ Final evaluation results with bias correction
- ✅ Brief analysis (1-2 paragraphs)

### Next Steps:

Congratulations! You've completed the LLM-as-Judge homework!

To go deeper:
- **Try Option 2**: Generate more traces and do more labeling
- **Try Option 1**: Generate your own traces from scratch
- **Explore other failure modes**: Apply this to different evaluation criteria
- **Read the course materials**: Sections 5.1-5.4 cover LLM-as-Judge in depth